# Project 37 — LangGraph Travel Planner Flow
## Gather preferences, plan itinerary, revise with checkpoints

**Stack:** LangGraph · Ollama · Jupyter

**Workflow:** Gather Prefs → Plan Itinerary → Review Budget → Finalize Plan

In [ ]:
# !pip install -q langgraph langchain langchain-ollama

## Step 1 — Setup

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:8b", temperature=0.2)

class WorkflowState(TypedDict):
    input_data: str
    accumulated: str
    final_output: str

## Step 2 — Define Workflow Nodes

In [ ]:
def gather_prefs(state: WorkflowState) -> WorkflowState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are step 1 of a LangGraph Travel Planner Flow pipeline. "
         "Your role: gather prefs. "
         "Process the input and produce structured output for the next step."),
        ("human", "Input: {input}\nPrevious steps: {context}")
    ])
    chain = prompt | llm | StrOutputParser()
    result = chain.invoke({"input": state["input_data"], "context": state.get("accumulated", "")})
    accumulated = state.get("accumulated", "") + f"\n\n[gather_prefs]: " + result
    print(f"  Step 1/4: gather_prefs")
    return {"accumulated": accumulated, "final_output": result}

def plan_itinerary(state: WorkflowState) -> WorkflowState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are step 2 of a LangGraph Travel Planner Flow pipeline. "
         "Your role: plan itinerary. "
         "Process the input and produce structured output for the next step."),
        ("human", "Input: {input}\nPrevious steps: {context}")
    ])
    chain = prompt | llm | StrOutputParser()
    result = chain.invoke({"input": state["input_data"], "context": state.get("accumulated", "")})
    accumulated = state.get("accumulated", "") + f"\n\n[plan_itinerary]: " + result
    print(f"  Step 2/4: plan_itinerary")
    return {"accumulated": accumulated, "final_output": result}

def review_budget(state: WorkflowState) -> WorkflowState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are step 3 of a LangGraph Travel Planner Flow pipeline. "
         "Your role: review budget. "
         "Process the input and produce structured output for the next step."),
        ("human", "Input: {input}\nPrevious steps: {context}")
    ])
    chain = prompt | llm | StrOutputParser()
    result = chain.invoke({"input": state["input_data"], "context": state.get("accumulated", "")})
    accumulated = state.get("accumulated", "") + f"\n\n[review_budget]: " + result
    print(f"  Step 3/4: review_budget")
    return {"accumulated": accumulated, "final_output": result}

def finalize_plan(state: WorkflowState) -> WorkflowState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are step 4 of a LangGraph Travel Planner Flow pipeline. "
         "Your role: finalize plan. "
         "Process the input and produce structured output for the next step."),
        ("human", "Input: {input}\nPrevious steps: {context}")
    ])
    chain = prompt | llm | StrOutputParser()
    result = chain.invoke({"input": state["input_data"], "context": state.get("accumulated", "")})
    accumulated = state.get("accumulated", "") + f"\n\n[finalize_plan]: " + result
    print(f"  Step 4/4: finalize_plan")
    return {"accumulated": accumulated, "final_output": result}

## Step 3 — Build and Compile Graph

In [ ]:
graph = StateGraph(WorkflowState)
graph.add_node("gather_prefs", gather_prefs)
graph.add_node("plan_itinerary", plan_itinerary)
graph.add_node("review_budget", review_budget)
graph.add_node("finalize_plan", finalize_plan)

graph.set_entry_point("gather_prefs")
graph.add_edge("gather_prefs", "plan_itinerary")
graph.add_edge("plan_itinerary", "review_budget")
graph.add_edge("review_budget", "finalize_plan")
graph.add_edge("finalize_plan", END)

app = graph.compile()
print("LangGraph Travel Planner Flow — workflow compiled!")

## Step 4 — Run the Workflow

In [ ]:
sample_input = "Analyze and process this request through the LangGraph Travel Planner Flow pipeline."
result = app.invoke({
    "input_data": sample_input, "accumulated": "", "final_output": "",
})

print("=== WORKFLOW RESULT ===")
print(result["final_output"])
print("\n=== FULL TRACE ===")
print(result["accumulated"][:1000])

## What You Learned
- **Multi-node LangGraph workflow** for gather preferences, plan itinerary, revise with checkpoints
- **Progressive context accumulation** across steps
- **Structured pipeline execution** with trace logging